# Vector stores and retrievers

This notebook section introduces LangChain's core abstractions for vector stores and retrievers, which facilitate retrieving context from vector databases or external sources to power LLM workflows and retrieval-augmented generation (RAG).

We will cover:

* Documents
* Vector stores
* Retrievers

In [9]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize embedding model (using standard sentence-transformers identifier)
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

e:\LangChain\venv\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2452.54it/s]


In [12]:
# VectorStores
vectorstore=Chroma.from_documents(documents, embedding=embedding)
vectorstore

In [13]:
vectorstore.similarity_search("cat")

[Document(id='3770aeb6-f28c-4188-bd8a-d7ae0dd5d241', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='42b452fe-6ce3-4457-84f2-88e1c33e9ef9', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='765d747f-f777-433e-a174-9124e91e34e0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [14]:
# Async Query
await vectorstore.asimilarity_search("cat")

[Document(id='3770aeb6-f28c-4188-bd8a-d7ae0dd5d241', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='42b452fe-6ce3-4457-84f2-88e1c33e9ef9', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='765d747f-f777-433e-a174-9124e91e34e0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [15]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='3770aeb6-f28c-4188-bd8a-d7ae0dd5d241', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351059198379517),
 (Document(id='42b452fe-6ce3-4457-84f2-88e1c33e9ef9', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351059198379517),
 (Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='765d747f-f777-433e-a174-9124e91e34e0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956)]

# Retrievers

- **VectorStore Integration:** LangChain `VectorStore` objects do not directly extend `Runnable`, meaning they cannot be chained immediately inside LangChain Expression Language (LCEL) workflows without an intermediate interface.
- **Retrievers as Runnables:** LangChain `Retriever` components inherit from `Runnable`. Consequently, they natively support core execution interfaces (such as synchronous/asynchronous `invoke` and batch operations) for seamless integration into LCEL pipelines.
- **Custom Runnable Construction:** You can construct a custom runnable wrapper around a vector store without extending the base `Retriever` class. By wrapping a specific retrieval mechanism—such as `similarity_search`—you can convert standard vector search into an LCEL-compatible component.

In [17]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='42b452fe-6ce3-4457-84f2-88e1c33e9ef9', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [18]:
vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

retriever.batch(["cat","Dog"])

[[Document(id='42b452fe-6ce3-4457-84f2-88e1c33e9ef9', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("tell me about dogs")
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "tell me about dogs"
   - **Constraint:** "Answer this question using the provided context only."
   - **Context:** `[Document(id='9b0f149f-e619-423d-8851-b482a589b9a3', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]`

2.  **Extract Information from Context:**
   - The context states: "Dogs are great companions, known for their loyalty and friendliness."

3.  **Formulate Answer based ONLY on Context:**
   - I need to directly use the provided sentence or paraphrase it closely without adding external knowledge.
   - Draft: Based on the provided context, dogs are great companions known for their loyalty and friendliness.

4.  **Check Constraints:**
   - Does it answer the question? Yes.
   - Is it based ONLY on the context? Yes.
   - Does it avoid external knowledge? Yes.

5.  **Final Output Generation:** (Keep it concise and d